第四章 智能体经典范式构建

4.1.3 封装基础 LLM 调用函数
为了让代码结构更清晰、更易于复用，我们来定义一个专属的LLM客户端类。这个类将封装所有与模型服务交互的细节，让我们的主逻辑可以更专注于智能体的构建。

In [1]:
import os
from openai import OpenAI
from typing import List, Dict

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

class HelloAgentsLLM:
    """
    为本书 "Hello Agents" 定制的LLM客户端。
    它用于调用任何兼容OpenAI接口的服务，并默认使用流式响应。
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化客户端。优先使用传入参数，如果未提供，则从环境变量加载。
        """
        self.model = model or os.getenv("OPENAI_MODEL_NAME")
        apiKey = apiKey or os.getenv("OPENAI_API_KEY")
        baseUrl = baseUrl or os.getenv("OPENAI_BASE_URL")
        timeout = timeout or int(os.getenv("OPENAI_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型进行思考，并返回其响应。
        """
        print(f"🧠 正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print("✅ 大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f"❌ 调用LLM API时发生错误: {e}")
            return None

# --- 客户端使用示例 ---
if __name__ == '__main__':
    try:
        llmClient = HelloAgentsLLM()
        
        exampleMessages = [
            {"role": "system", "content": "You are a helpful assistant that writes Python code."},
            {"role": "user", "content": "写一个快速排序算法"}
        ]
        
        print("--- 调用LLM ---")
        responseText = llmClient.think(exampleMessages)
        if responseText:
            print("\n\n--- 完整模型响应 ---")
            print(responseText)

    except ValueError as e:
        print(e)


--- 调用LLM ---
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
以下是使用 Python 实现的快速排序算法，包括递归版本、原地优化版本以及非递归版本，并附有测试用例与复杂度分析，适合学习和实际应用参考。

---

## 1. 基础递归版本（返回新列表）

```python
def quick_sort(arr):
    """
    快速排序 - 基础递归版本（返回新列表）
    时间复杂度: O(n log n) 平均, O(n²) 最坏
    空间复杂度: O(n)
    """
    if len(arr) <= 1:
        return arr
    
    pivot = arr[len(arr) // 2]  # 选择中间元素作为基准
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    
    return quick_sort(left) + middle + quick_sort(right)
```

---

## 2. 原地快速排序（推荐）

```python
def quick_sort_inplace(arr, low=0, high=None):
    """
    快速排序 - 原地递归版本（修改原数组）
    时间复杂度: O(n log n) 平均, O(n²) 最坏
    空间复杂度: O(log n)
    """
    if high is None:
        high = len(arr) - 1
    
    if low < high:
        pivot_index = partition(arr, low, high)
        quick_sort_inplace(arr, low, pivot_index - 1)
        quick_sort_inplace(arr, pivot_index + 1, high)
    
    return arr

def pa

### 4.2.2 工具的定义与实现

如果说大语言模型是智能体的大脑，那么<strong>工具 (Tools)</strong> 就是其与外部世界交互的“手和脚”。为了让ReAct范式能够真正解决我们设定的问题，智能体需要具备调用外部工具的能力。

针对本节设定的目标——回答关于“华为最新手机”的问题，我们需要为智能体提供一个网页搜索工具。在这里我们选用 <strong>SerpApi</strong>，它通过API提供结构化的Google搜索结果，能直接返回“答案摘要框”或精确的知识图谱信息。

首先，需要安装该库：

```bash
pip install google-search-results
```

同时，你需要前往 [SerpApi官网](https://serpapi.com/) 注册一个免费账户，获取你的API密钥，并将其添加到我们项目根目录下的 `.env` 文件中：

```bash
# .env file
# ... (保留之前的LLM配置)
SERPAPI_API_KEY="YOUR_SERPAPI_API_KEY"
```

接下来，我们通过代码来定义和管理这个工具。我们将分步进行：首先实现工具的核心功能，然后构建一个通用的工具管理器。

（1）实现搜索工具的核心逻辑

一个良好定义的工具应包含以下三个核心要素：

1. <strong>名称 (Name)</strong>： 一个简洁、唯一的标识符，供智能体在 `Action` 中调用，例如 `Search`。
2. <strong>描述 (Description)</strong>： 一段清晰的自然语言描述，说明这个工具的用途。<strong>这是整个机制中最关键的部分</strong>，因为大语言模型会依赖这段描述来判断何时使用哪个工具。
3. <strong>执行逻辑 (Execution Logic)</strong>： 真正执行任务的函数或方法。

我们的第一个工具是 `search` 函数，它的作用是接收一个查询字符串，然后返回搜索结果。

In [2]:
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    一个基于SerpApi的实战网页搜索引擎工具。
    它会智能地解析搜索结果，优先返回直接答案或知识图谱信息。
    """
    print(f"🔍 正在执行 [SerpApi] 网页搜索: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误:SERPAPI_API_KEY 未在 .env 文件中配置。"

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # 国家代码
            "hl": "zh-cn", # 语言代码
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # 智能解析:优先寻找最直接的答案
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"

In [3]:
print(search("华为手机"))

🔍 正在执行 [SerpApi] 网页搜索: 华为手机
华为是总部位于中华人民共和国广东省深圳市的科技公司，业务以研发和制造通信设备、消费电子产品为主，除此之外还涉足软件开发、设计生产集成电路、光伏和电动汽车等跨界产品。


（2）构建通用的工具执行器

当智能体需要使用多种工具时（例如，除了搜索，还可能需要计算、查询数据库等），我们需要一个统一的管理器来注册和调度这些工具。为此，我们创建一个 ToolExecutor 类。

In [4]:
from typing import Dict, Any

class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])


In [5]:
# 1.初始化工具执行器
tool_executor = ToolExecutor()
# 2.注册我们的实战搜索工具
search_description = "一个网页的搜索引擎。当你需要回答关于事实、时事以及你的数据库中的信息时，应使用此工具。"
#                          工具名称      工具描述      工具函数
tool_executor.registerTool("Search", search_description, search)
#3.打印所有可用工具
print(tool_executor.getAvailableTools())
# 4.智能体的Action调用，这次我们问一个实时性的问题
print("\n\n\n--- 执行Action: Search['英伟达最新的CPU型号是什么...']")
tool_name = "Search"
tool_input = "英伟达最新的CPU型号是什么？"
tool_function = tool_executor.getTool(tool_name)
if tool_function:
    observation = tool_function(tool_input) # 执行搜索工具
    print("-----观察(observation)-------")
    print(observation)
else:
    print(f"错误:未找到工具 '{tool_name}'。请检查工具名称是否正确。")


工具 'Search' 已注册。
- Search: 一个网页的搜索引擎。当你需要回答关于事实、时事以及你的数据库中的信息时，应使用此工具。



--- 执行Action: Search['英伟达最新的CPU型号是什么...']
🔍 正在执行 [SerpApi] 网页搜索: 英伟达最新的CPU型号是什么？
-----观察(observation)-------
[1] NVIDIA Grace CPU 和ARM 架构
Grace CPU 超级芯片由两个Grace CPU 组成，这两个CPU 通过NVIDIA NVLink-C2C 以900 GB/s 的速度耦合连接。它将144 个Arm Neoverse V2 核心封装到单个模块中，并配备服务器 ...

[2] NVIDIA Grace CPU 超级芯片
NVIDIA Grace CPU 产品组合中包括Grace 超级芯片，它是紧凑型双路服务器的核心模块，集成144 个Neoverse V2 核心与最高达960 GB 的LPDDR5X，CPU 和内存总功耗仅为500 W。

[3] NVIDIA 正式发布Vera：专为智能体打造的CPU
Vera 通过第二代NVIDIA NVLink™-C2C 互连技术，担任NVIDIA Vera Rubin 平台的主机CPU。该技术支持CPU 与GPU 之间高达1.8TB/s 的相干带宽，并将NVIDIA 机密 ...


### 4.2.3 ReAct 智能体的编码实现
现在，我们将所有独立的组件，LLM客户端和工具执行器组装起来，构建一个完整的 ReAct 智能体。我们将通过一个 ReActAgent 类来封装其核心逻辑。为了便于理解，我们将这个类的实现过程拆分为以下几个关键部分进行讲解。

（1）系统提示词设计

提示词是整个 ReAct 机制的基石，它为大语言模型提供了行动的操作指令。我们需要精心设计一个模板，它将动态地插入可用工具、用户问题以及中间步骤的交互历史。

In [6]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

这个模板定义了智能体与LLM之间交互的规范：

角色定义： “你是一个有能力调用外部工具的智能助手”，设定了LLM的角色。
工具清单 ({tools})： 告知LLM它有哪些可用的“手脚”。
格式规约 (Thought/Action)： 这是最重要的部分，它强制LLM的输出具有结构性，使我们能通过代码精确解析其意图。
动态上下文 ({question}/{history})： 将用户的原始问题和不断累积的交互历史注入，让LLM基于完整的上下文进行决策。
（2）核心循环的实现

ReActAgent 的核心是一个循环，它不断地“格式化提示词 -> 调用LLM -> 执行动作 -> 整合结果”，直到任务完成或达到最大步数限制。

（3）输出解析器的实现

LLM 返回的是纯文本，我们需要从中精确地提取出Thought和Action。这是通过几个辅助解析函数完成的，它们通常使用正则表达式来实现。

In [7]:
import re
from typing import Optional, List, Dict, Callable

class ReActAgent:
    def __init__(self, llm_client: "HelloAgentsLLM", tool_executor: "ToolExecutor", max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []  # 存储 (thought, action, observation) 或简单字符串

    def run(self, question: str) -> Optional[str]:
        """
        运行 ReAct 智能体回答用户问题，返回最终答案字符串，若失败返回 None。
        """
        self.history = []  # 重置
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"\n--- 第 {current_step} 步 ---")

            # 1. 格式化提示词
            tools_desc = self.tool_executor.getAvailableTools()  # 假设返回字符串描述
            history_str = self._format_history()  # 自定义历史格式化
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. 调用 LLM
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)
            if not response_text:
                print("❌ LLM 未返回有效响应，终止。")
                break

            # 3. 解析 Thought 和 Action
            thought, action = self._parse_output(response_text)
            if thought:
                print(f"🧠 思考: {thought}")
            else:
                print("⚠️ 未能解析出 Thought，继续...")

            # 记录历史（包含思考，便于上下文）
            self.history.append({"role": "thought", "content": thought or "无思考"})

            if not action:
                # 若没有 Action，尝试引导模型重新生成
                observation = "错误：没有提供有效的 Action。请重新输出正确的 Thought 和 Action。"
                self.history.append({"role": "observation", "content": observation})
                print(f"👀 观察: {observation}")
                continue  # 继续下一轮，让模型重试

            # 4. 处理 Finish
            if action.startswith("Finish"):
                # 提取最终答案
                finish_match = re.match(r"Finish\[(.*)\]", action, re.DOTALL)
                if finish_match:
                    final_answer = finish_match.group(1).strip()
                    print(f"🎉 最终答案: {final_answer}")
                    return final_answer
                else:
                    # 格式错误，给予反馈
                    observation = "错误：Finish 格式应为 Finish[答案]，请修正。"
                    self.history.append({"role": "observation", "content": observation})
                    print(f"👀 观察: {observation}")
                    continue

            # 5. 解析工具名称和输入
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or tool_input is None:
                observation = f"错误：无法解析 Action 格式，期望 '工具名[输入]'，得到 '{action}'"
                self.history.append({"role": "observation", "content": observation})
                print(f"👀 观察: {observation}")
                continue

            print(f"🎬 行动: {tool_name}[{tool_input}]")

            # 6. 执行工具
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误：未找到名为 '{tool_name}' 的工具。"
            else:
                try:
                    observation = tool_function(tool_input)
                except Exception as e:
                    observation = f"执行工具时出错: {e}"

            print(f"👀 观察: {observation}")

            # 7. 记录本轮 Action 和 Observation
            self.history.append({"role": "action", "content": action})
            self.history.append({"role": "observation", "content": observation})

        # 超出最大步数
        print(f"⚠️ 已达到最大步数 {self.max_steps}，流程终止。")
        return None

    # ----- 辅助方法 -----

    def _format_history(self) -> str:
        """将历史记录格式化为字符串，供提示词使用"""
        lines = []
        for entry in self.history:
            role = entry.get("role")
            content = entry.get("content", "")
            if role == "thought":
                lines.append(f"Thought: {content}")
            elif role == "action":
                lines.append(f"Action: {content}")
            elif role == "observation":
                lines.append(f"Observation: {content}")
        return "\n".join(lines)

    def _parse_output(self, text: str):
        """
        从 LLM 输出中提取 Thought 和 Action。
        返回 (thought, action) 元组，若未找到则为 (None, None)。
        """
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None

        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        action = action_match.group(1).strip() if action_match else None

        return thought, action

    def _parse_action(self, action_text: str):
        """
        解析 Action 字符串，返回 (tool_name, tool_input)。
        格式: tool_name[tool_input]
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None

(4) 工具调用与执行

In [8]:
llm = HelloAgentsLLM()
tool_executor = ToolExecutor()
search_desc = "一个网页的搜索引擎。当你需要回答关于事实、时事以及你的数据库中的信息时，应使用此工具。"
tool_executor.registerTool("Search", search_desc, search)

agent = ReActAgent(llm_client=llm, tool_executor=tool_executor)

# 给一个问题
question = "马克斯的optimus机器人目前产量状态?它的中国供应链公司有哪些?"
agent.run(question)


工具 'Search' 已注册。

--- 第 1 步 ---
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
Thought: 用户询问的是特斯拉Optimus机器人的产量状态和中国供应链公司信息。这是一个需要最新事实信息的问题，我应该使用Search工具来搜索相关数据。

Action: Search[特斯拉Optimus机器人 2024年产量状态]
🧠 思考: 用户询问的是特斯拉Optimus机器人的产量状态和中国供应链公司信息。这是一个需要最新事实信息的问题，我应该使用Search工具来搜索相关数据。
🎬 行动: Search[特斯拉Optimus机器人 2024年产量状态]
🔍 正在执行 [SerpApi] 网页搜索: 特斯拉Optimus机器人 2024年产量状态
👀 观察: [1] 特斯拉人形机器人陷入窘境：产量大幅落后技术未达预期
马斯克曾承诺，特斯拉将在2025年至少生产5000台Optimus人形机器人。然而 ... 2021年，他又表示，特斯拉将在2024年实现自动驾驶出租车的量产。但 ...

[2] 每年翻10倍！马斯克：2026年特斯拉人形机器人产量将增加 ...
Optimus将在2024年底- 2025年初硬件改版，2025年限量生产，或有超1000台试运行。其价格在2 - 3万美元。人形机器人赛道竞争激烈，国内外众多企业涉足 ...

[3] 特斯拉第三代人形机器人预计年中发布，三季度启动正式投产
... 2024年进入工厂测试。 特斯拉在2026年第一季度财报中披露称，Optimus第一代生产线设计年产100万台机器人，将取代位于弗里蒙特的Model S和Model X生产线。

--- 第 2 步 ---
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
Thought: 我已经获取了关于Optimus机器人产量状态的部分信息，包括2024-2025年的生产计划和目标。但我还需要补充关于中国供应链公司的具体信息，这是用户问题的另一个重要部分。我需要再次使用Search工具来搜索相关信息。

Action: Search[特斯拉Optimus机器人 中国供应链公司]
🧠 思考: 我已经获取了关于Optimus机器人产量状态的部分信息，包括20

'特斯拉Optimus机器人目前产量状态及中国供应链公司信息如下：\n\n**一、产量状态：**\n- 2024年：进入工厂测试阶段\n- 2024年底-2025年初：进行硬件改版\n- 2025年：限量生产，预计超1000台试运行；马斯克承诺至少生产5000台\n- 2026年：第一代生产线设计年产100万台机器人\n\n**二、中国供应链公司（共7家，占半壁江山）：**\n核心受益标的包括：\n1. **三花智控** - 核心执行器总成供应商，供应旋转关节与线性关节总成\n2. **拓普集团** - 确定性最高、价值量最大的核心供应商\n3. **绿的谐波** - 确定性最高、价值量最大的核心供应商\n\n其他中国供应商还包括多家企业，共同构成Optimus供应链的重要部分。'

### 4.2.4 ReAct 的特点、局限性与调试技巧

通过亲手实现一个 ReAct 智能体，我们不仅掌握了其工作流程，也应该对其内在机制有了更深刻的认识。任何技术范式都有其闪光点和待改进之处，本节将对 ReAct 进行总结。

（1）ReAct 的主要特点

1. <strong>高可解释性</strong>：ReAct 最大的优点之一就是透明。通过 `Thought` 链，我们可以清晰地看到智能体每一步的“心路历程”——它为什么会选择这个工具，下一步又打算做什么。这对于理解、信任和调试智能体的行为至关重要。
2. <strong>动态规划与纠错能力</strong>：与一次性生成完整计划的范式不同，ReAct 是“走一步，看一步”。它根据每一步从外部世界获得的 `Observation` 来动态调整后续的 `Thought` 和 `Action`。如果上一步的搜索结果不理想，它可以在下一步中修正搜索词，重新尝试。
3. <strong>工具协同能力</strong>：ReAct 范式天然地将大语言模型的推理能力与外部工具的执行能力结合起来。LLM 负责运筹帷幄（规划和推理），工具负责解决具体问题（搜索、计算），二者协同工作，突破了单一 LLM 在知识时效性、计算准确性等方面的固有局限。

（2）ReAct 的固有局限性

1. <strong>对LLM自身能力的强依赖</strong>：ReAct 流程的成功与否，高度依赖于底层 LLM 的综合能力。如果 LLM 的逻辑推理能力、指令遵循能力或格式化输出能力不足，就很容易在 `Thought` 环节产生错误的规划，或者在 `Action` 环节生成不符合格式的指令，导致整个流程中断。
2. <strong>执行效率问题</strong>：由于其循序渐进的特性，完成一个任务通常需要多次调用 LLM。每一次调用都伴随着网络延迟和计算成本。对于需要很多步骤的复杂任务，这种串行的“思考-行动”循环可能会导致较高的总耗时和费用。
3. <strong>提示词的脆弱性</strong>：整个机制的稳定运行建立在一个精心设计的提示词模板之上。模板中的任何微小变动，甚至是用词的差异，都可能影响 LLM 的行为。此外，并非所有模型都能持续稳定地遵循预设的格式，这增加了在实际应用中的不确定性。
4. <strong>可能陷入局部最优</strong>：步进式的决策模式意味着智能体缺乏一个全局的、长远的规划。它可能会因为眼前的 `Observation` 而选择一个看似正确但长远来看并非最优的路径，甚至在某些情况下陷入“原地打转”的循环中。

（3）调试技巧

当你构建的 ReAct 智能体行为不符合预期时，可以从以下几个方面入手进行调试：

- <strong>检查完整的提示词</strong>：在每次调用 LLM 之前，将最终格式化好的、包含所有历史记录的完整提示词打印出来。这是追溯 LLM 决策源头的最直接方式。
- <strong>分析原始输出</strong>：当输出解析失败时（例如，正则表达式没有匹配到 `Action`），务必将 LLM 返回的原始、未经处理的文本打印出来。这能帮助你判断是 LLM 没有遵循格式，还是你的解析逻辑有误。
- <strong>验证工具的输入与输出</strong>：检查智能体生成的 `tool_input` 是否是工具函数所期望的格式，同时也要确保工具返回的 `observation` 格式是智能体可以理解和处理的。
- <strong>调整提示词中的示例 (Few-shot Prompting)</strong>：如果模型频繁出错，可以在提示词中加入一两个完整的“Thought-Action-Observation”成功案例，通过示例来引导模型更好地遵循你的指令。
- <strong>尝试不同的模型或参数</strong>：更换一个能力更强的模型，或者调整 `temperature` 参数（通常设为0以保证输出的确定性），有时能直接解决问题。


## 4.3 Plan-and-Solve

在我们掌握了 ReAct 这种反应式的、步进决策的智能体范式后，接下来将探讨一种风格迥异但同样强大的方法，<strong>Plan-and-Solve</strong>。顾名思义，这种范式将任务处理明确地分为两个阶段：<strong>先规划 (Plan)，后执行 (Solve)</strong>。

如果说 ReAct 像一个经验丰富的侦探，根据现场的蛛丝马迹（Observation）一步步推理，随时调整自己的调查方向；那么 Plan-and-Solve 则更像一位建筑师，在动工之前必须先绘制出完整的蓝图（Plan），然后严格按照蓝图来施工（Solve）。事实上我们现在用的很多大模型工具的Agent模式都融入了这种设计模式。

### 4.3.1 Plan-and-Solve 的工作原理

Plan-and-Solve Prompting 由 Lei Wang 在2023年提出<sup>[2]</sup>。其核心动机是为了解决思维链在处理多步骤、复杂问题时容易“偏离轨道”的问题。

与 ReAct 将思考和行动融合在每一步不同，Plan-and-Solve 将整个流程解耦为两个核心阶段，如图4.2所示：

1. <strong>规划阶段 (Planning Phase)</strong>： 首先，智能体会接收用户的完整问题。它的第一个任务不是直接去解决问题或调用工具，而是<strong>将问题分解，并制定出一个清晰、分步骤的行动计划</strong>。这个计划本身就是一次大语言模型的调用产物。
2. <strong>执行阶段 (Solving Phase)</strong>： 在获得完整的计划后，智能体进入执行阶段。它会<strong>严格按照计划中的步骤，逐一执行</strong>。每一步的执行都可能是一次独立的 LLM 调用，或者是对上一步结果的加工处理，直到计划中的所有步骤都完成，最终得出答案。

这种“先谋后动”的策略，使得智能体在处理需要长远规划的复杂任务时，能够保持更高的目标一致性，避免在中间步骤中迷失方向。

我们可以将这个两阶段过程进行形式化表达。首先，规划模型 $\pi_{\text{plan}}$ 根据原始问题 $q$ 生成一个包含 $n$ 个步骤的计划 $P = (p_1, p_2, \dots, p_n)$：

$$
P = \pi_{\text{plan}}(q)
$$

随后，在执行阶段，执行模型 $\pi_{\text{solve}}$ 会逐一完成计划中的步骤。对于第 $i$ 个步骤，其解决方案 $s_i$ 的生成会同时依赖于原始问题 $q$、完整计划 $P$ 以及之前所有步骤的执行结果 $(s_1, \dots, s_{i-1})$：

$$
s_i = \pi_{\text{solve}}(q, P, (s_1, \dots, s_{i-1}))
$$

最终的答案就是最后一个步骤的执行结果 $s_n$。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/4-figures/4-2.png" alt="Plan-and-Solve范式的两阶段工作流" width="90%"/>
  <p>图 4.2 Plan-and-Solve 范式的两阶段工作流</p>
</div>

Plan-and-Solve 尤其适用于那些结构性强、可以被清晰分解的复杂任务，例如：

- <strong>多步数学应用题</strong>：需要先列出计算步骤，再逐一求解。
- <strong>需要整合多个信息源的报告撰写</strong>：需要先规划好报告结构（引言、数据来源A、数据来源B、总结），再逐一填充内容。
- <strong>代码生成任务</strong>：需要先构思好函数、类和模块的结构，再逐一实现。

### 4.3.2 规划阶段

为了凸显 Plan-and-Solve 范式在结构化推理任务上的优势，我们将不使用工具的方式，而是通过提示词的设计，完成一个推理任务。

这类任务的特点是，答案无法通过单次查询或计算得出，必须先将问题分解为一系列逻辑连贯的子步骤，然后按顺序求解。这恰好能发挥 Plan-and-Solve “先规划，后执行”的核心能力。

<strong>我们的目标问题是：</strong>“一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？”

这个问题对于大语言模型来说并不算特别困难，但它包含了一个清晰的逻辑链条可供参考。在某些实际的逻辑难题上，如果大模型不能高质量的推理出准确的答案，可以参考这个设计模式来设计自己的Agent完成任务。智能体需要：

1. <strong>规划阶段</strong>：首先，将问题分解为三个独立的计算步骤（计算周二销量、计算周三销量、计算总销量）。
2. <strong>执行阶段</strong>：然后，严格按照计划，一步步执行计算，并将每一步的结果作为下一步的输入，最终得出总和。

规划阶段的目标是让大语言模型接收原始问题，并输出一个清晰、分步骤的行动计划。这个计划必须是结构化的，以便我们的代码可以轻松解析并逐一执行。因此，我们设计的提示词需要明确地告诉模型它的角色和任务，并给出一个输出格式的范例。


In [9]:
PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是一个独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划,```python与```作为前后缀是必要的:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""

这个提示词通过以下几点确保了输出的质量和稳定性：
- <strong>角色设定</strong>： “顶级的AI规划专家”，激发模型的专业能力。
- <strong>任务描述</strong>： 清晰地定义了“分解问题”的目标。
- <strong>格式约束</strong>： 强制要求输出为一个 Python 列表格式的字符串，这极大地简化了后续代码的解析工作，使其比解析自然语言更稳定、更可靠。

接下来，我们将这个提示词逻辑封装成一个 `Planner` 类，这个类也是我们的规划器。

In [10]:
# 假定 llm_client.py 中的 HelloAgentsLLM 类已经定义好
# from llm_client import HelloAgentsLLM
import ast

class Planner:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def plan(self, question: str) -> list[str]:
        """
        根据用户问题生成一个行动计划。
        """
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)
        
        # 为了生成计划，我们构建一个简单的消息列表
        messages = [{"role": "user", "content": prompt}]
        
        print("--- 正在生成计划 ---")
        # 使用流式输出来获取完整的计划
        response_text = self.llm_client.think(messages=messages) or ""
        
        print(f"✅ 计划已生成:\n{response_text}")
        
        # 解析LLM输出的列表字符串
        try:
            # 找到```python和```之间的内容
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            # 使用ast.literal_eval来安全地执行字符串，将其转换为Python列表
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"❌ 解析计划时出错: {e}")
            print(f"原始响应: {response_text}")
            return []
        except Exception as e:
            print(f"❌ 解析计划时发生未知错误: {e}")
            return []

In [11]:
try:
    llm = HelloAgentsLLM()
    planner = Planner(llm)
    question = "一个水果店周一卖出15个苹果。周二卖出的苹果是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？"
    list_plan = planner.plan(question)
    print(list_plan, "\n==================\n", type(list_plan))
except Exception as e:
    print(f"❌ 初始化LLM客户端时出错: {e}")
    raise e

--- 正在生成计划 ---
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
```python
[
    "步骤1：确定周一卖出的苹果数量为15个",
    "步骤2：计算周二卖出的苹果数量（周一数量乘以2）",
    "步骤3：计算周三卖出的苹果数量（周二数量减去5）",
    "步骤4：将周一、周二和周三的苹果数量相加得到总销量"
]
```
✅ 计划已生成:
```python
[
    "步骤1：确定周一卖出的苹果数量为15个",
    "步骤2：计算周二卖出的苹果数量（周一数量乘以2）",
    "步骤3：计算周三卖出的苹果数量（周二数量减去5）",
    "步骤4：将周一、周二和周三的苹果数量相加得到总销量"
]
```
['步骤1：确定周一卖出的苹果数量为15个', '步骤2：计算周二卖出的苹果数量（周一数量乘以2）', '步骤3：计算周三卖出的苹果数量（周二数量减去5）', '步骤4：将周一、周二和周三的苹果数量相加得到总销量'] 
 <class 'list'>


4.3.3 执行器与状态管理

在规划器 (Planner) 生成了清晰的行动蓝图后，我们就需要一个执行器 (Executor) 来逐一完成计划中的任务。执行器不仅负责调用大语言模型来解决每个子问题，还承担着一个至关重要的角色：状态管理。它必须记录每一步的执行结果，并将其作为上下文提供给后续步骤，确保信息在整个任务链条中顺畅流动

执行器的提示词与规划器不同。它的目标不是分解问题，而是在已有上下文的基础上，专注解决当前这一个步骤。因此，提示词需要包含以下关键信息：

原始问题： 确保模型始终了解最终目标。
完整计划： 让模型了解当前步骤在整个任务中的位置。
历史步骤与结果： 提供至今为止已经完成的工作，作为当前步骤的直接输入。
当前步骤： 明确指示模型现在需要解决哪一个具体任务。

In [12]:
EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决“当前步骤”，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对“当前步骤”的回答:
"""

我们将执行逻辑封装到 Executor 类中。这个类将循环遍历计划，调用 LLM，并维护一个历史记录（状态）。

In [13]:
class Executor:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def execute(self, question: str, plan: list[str]) -> str:
        """
        根据计划，逐步执行并解决问题。
        """
        history = "" # 用于存储历史步骤和结果的字符串
        
        print("\n--- 正在执行计划 ---")
        
        for i, step in enumerate(plan):
            print(f"\n-> 正在执行步骤 {i+1}/{len(plan)}: {step}")
            
            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "无", # 如果是第一步，则历史为空
                current_step=step
            )
            
            messages = [{"role": "user", "content": prompt}]
            
            response_text = self.llm_client.think(messages=messages) or ""
            
            # 更新历史记录，为下一步做准备
            history += f"步骤 {i+1}: {step}\n结果: {response_text}\n\n"
            
            print(f"✅ 步骤 {i+1} 已完成，结果: {response_text}")

        # 循环结束后，最后一步的响应就是最终答案
        final_answer = response_text
        return final_answer

现在已经分别构建了负责“规划”的 Planner 和负责“执行”的 Executor。最后一步是将这两个组件整合到一个统一的智能体 PlanAndSolveAgent 中，并赋予它解决问题的完整能力。我们将创建一个主类 PlanAndSolveAgent，它的职责非常清晰：接收一个 LLM 客户端，初始化内部的规划器和执行器，并提供一个简单的 run 方法来启动整个流程。

In [14]:
class PlanAndSolveAgent:
    def __init__(self, llm_client):
        """
        初始化智能体，同时创建规划器和执行器实例。
        """
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)

    def run(self, question: str):
        """
        运行智能体的完整流程:先规划，后执行。
        """
        print(f"\n--- 开始处理问题 ---\n问题: {question}")
        
        # 1. 调用规划器生成计划
        plan = self.planner.plan(question)
        
        # 检查计划是否成功生成
        if not plan:
            print("\n--- 任务终止 --- \n无法生成有效的行动计划。")
            return

        # 2. 调用执行器执行计划
        final_answer = self.executor.execute(question, plan)
        
        print(f"\n--- 任务完成 ---\n最终答案: {final_answer}")

这个 PlanAndSolveAgent 类的设计体现了“组合优于继承”的原则。它本身不包含复杂的逻辑，而是作为一个协调者 (Orchestrator)，清晰地调用其内部组件来完成任务。

In [15]:
try:
    llm = HelloAgentsLLM()
    agent = PlanAndSolveAgent(llm)
    question = "一个水果店周一卖出15个苹果。周二卖出的苹果是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？"
    list_plan = agent.run(question)
    print(list_plan, "\n==================\n", type(list_plan))
except Exception as e:
    print(f"❌ 初始化LLM客户端时出错: {e}")
    raise e


--- 开始处理问题 ---
问题: 一个水果店周一卖出15个苹果。周二卖出的苹果是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？
--- 正在生成计划 ---
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
```python
[
    "步骤1：获取周一卖出的苹果数量（已知为15个）",
    "步骤2：计算周二卖出的苹果数量（周一数量乘以2）",
    "步骤3：计算周三卖出的苹果数量（周二数量减去5）",
    "步骤4：计算三天卖出的苹果总数（将周一、周二和周三的数量相加）"
]
```
✅ 计划已生成:
```python
[
    "步骤1：获取周一卖出的苹果数量（已知为15个）",
    "步骤2：计算周二卖出的苹果数量（周一数量乘以2）",
    "步骤3：计算周三卖出的苹果数量（周二数量减去5）",
    "步骤4：计算三天卖出的苹果总数（将周一、周二和周三的数量相加）"
]
```

--- 正在执行计划 ---

-> 正在执行步骤 1/4: 步骤1：获取周一卖出的苹果数量（已知为15个）
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
15 个
✅ 步骤 1 已完成，结果: 15 个

-> 正在执行步骤 2/4: 步骤2：计算周二卖出的苹果数量（周一数量乘以2）
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
结果：30 个
✅ 步骤 2 已完成，结果: 结果：30 个

-> 正在执行步骤 3/4: 步骤3：计算周三卖出的苹果数量（周二数量减去5）
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
25 个
✅ 步骤 3 已完成，结果: 25 个

-> 正在执行步骤 4/4: 步骤4：计算三天卖出的苹果总数（将周一、周二和周三的数量相加）
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
70 个
✅ 步骤 4 已完成，结果: 70 个

--- 任务完成 ---
最终答案: 70 个
None 
 <class 'NoneType'>


## 4.4 Reflection

在我们已经实现的 ReAct 和 Plan-and-Solve 范式中，智能体一旦完成了任务，其工作流程便告结束。然而，它们生成的初始答案，无论是行动轨迹还是最终结果，都可能存在谬误或有待改进之处。Reflection 机制的核心思想，正是为智能体引入一种<strong>事后（post-hoc）的自我校正循环</strong>，使其能够像人类一样，审视自己的工作，发现不足，并进行迭代优化。

### 4.4.1 Reflection 机制的核心思想

Reflection 机制的灵感来源于人类的学习过程：我们完成初稿后会进行校对，解出数学题后会进行验算。这一思想在多个研究中得到了体现，例如 Shinn, Noah 在2023年提出的 Reflexion 框架<sup>[3]</sup>。其核心工作流程可以概括为一个简洁的三步循环：<strong>执行 -> 反思 -> 优化</strong>。

1. <strong>执行 (Execution)</strong>：首先，智能体使用我们熟悉的方法（如 ReAct 或 Plan-and-Solve）尝试完成任务，生成一个初步的解决方案或行动轨迹。这可以看作是“初稿”。
2. <strong>反思 (Reflection)</strong>：接着，智能体进入反思阶段。它会调用一个独立的、或者带有特殊提示词的大语言模型实例，来扮演一个“评审员”的角色。这个“评审员”会审视第一步生成的“初稿”，并从多个维度进行评估，例如：
   - <strong>事实性错误</strong>：是否存在与常识或已知事实相悖的内容？
   - <strong>逻辑漏洞</strong>：推理过程是否存在不连贯或矛盾之处？
   - <strong>效率问题</strong>：是否有更直接、更简洁的路径来完成任务？
   - <strong>遗漏信息</strong>：是否忽略了问题的某些关键约束或方面？ 根据评估，它会生成一段结构化的<strong>反馈 (Feedback)</strong>，指出具体的问题所在和改进建议。
3. <strong>优化 (Refinement)</strong>：最后，智能体将“初稿”和“反馈”作为新的上下文，再次调用大语言模型，要求它根据反馈内容对初稿进行修正，生成一个更完善的“修订稿”。

如图4.3所示，这个循环可以重复进行多次，直到反思阶段不再发现新的问题，或者达到预设的迭代次数上限。我们可以将这个迭代优化的过程形式化地表达出来。假设 $O_i$ 是第 $i$ 次迭代产生的输出（$O_0$ 为初始输出），反思模型 $\pi_{\text{reflect}}$ 会生成针对 $O_i$ 的反馈 $F_i$：
$$
F_i = \pi_{\text{reflect}}(\text{Task}, O_i)
$$
随后，优化模型 $\pi_{\text{refine}}$ 会结合原始任务、上一版输出以及反馈，生成新一版的输出 $O_{i+1}$：
$$
O_{i+1} = \pi_{\text{refine}}(\text{Task}, O_i, F_i)
$$



<div align="center">
<img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/4-figures/4-3.png" alt="Reflection机制中的“执行-反思-优化”迭代循环" width="70%"/>
<p>图 4.3 Reflection 机制中的“执行-反思-优化”迭代循环</p>
</div>



与前两种范式相比，Reflection 的价值在于：

- 它为智能体提供了一个内部纠错回路，使其不再完全依赖于外部工具的反馈（ReAct 的 Observation），从而能够修正更高层次的逻辑和策略错误。
- 它将一次性的任务执行，转变为一个持续优化的过程，显著提升了复杂任务的最终成功率和答案质量。
- 它为智能体构建了一个临时的<strong>“短期记忆”</strong>。整个“执行-反思-优化”的轨迹形成了一个宝贵的经验记录，智能体不仅知道最终答案，还记得自己是如何从有缺陷的初稿迭代到最终版本的。更进一步，这个记忆系统还可以是<strong>多模态的</strong>，允许智能体反思和修正文本以外的输出（如代码、图像等），为构建更强大的多模态智能体奠定了基础。

### 4.4.2 案例设定与记忆模块设计

为了在实战中体现 Reflection 机制，我们将引入记忆管理机制，因为reflection通常对应着信息的存储和提取，如果上下文足够长的情况，想让“评审员”直接获取所有的信息然后进行反思往往会传入很多冗余信息。这一步实践我们主要完成<strong>代码生成与迭代优化</strong>。

这一步的目标任务是：“编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。”

这个任务是检验 Reflection 机制的绝佳场景：

1. <strong>存在明确的优化路径</strong>：大语言模型初次生成的代码很可能是一个简单但效率低下的递归实现。
2. <strong>反思点清晰</strong>：可以通过反思发现其“时间复杂度过高”或“存在重复计算”的问题。
3. <strong>优化方向明确</strong>：可以根据反馈，将其优化为更高效的迭代版本或使用备忘录模式的版本。

Reflection 的核心在于迭代，而迭代的前提是能够记住之前的尝试和获得的反馈。因此，一个“短期记忆”模块是实现该范式的必需品。这个记忆模块将负责存储每一次“执行-反思”循环的完整轨迹。


In [16]:
from typing import List, Dict, Any, Optional

class Memory:
    """
    一个简单的短期记忆模块，用于存储智能体的行动与反思轨迹。
    """

    def __init__(self):
        """
        初始化一个空列表来存储所有记录。
        """
        self.records: List[Dict[str, Any]] = []

    def add_record(self, record_type: str, content: str):
        """
        向记忆中添加一条新记录。

        参数:
        - record_type (str): 记录的类型 ('execution' 或 'reflection')。
        - content (str): 记录的具体内容 (例如，生成的代码或反思的反馈)。
        """
        record = {"type": record_type, "content": content}
        self.records.append(record)
        print(f"📝 记忆已更新，新增一条 '{record_type}' 记录。")

    def get_trajectory(self) -> str:
        """
        将所有记忆记录格式化为一个连贯的字符串文本，用于构建提示词。
        """
        trajectory_parts = []
        for record in self.records:
            if record['type'] == 'execution':
                trajectory_parts.append(f"--- 上一轮尝试 (代码) ---\n{record['content']}")
            elif record['type'] == 'reflection':
                trajectory_parts.append(f"--- 评审员反馈 ---\n{record['content']}")
        
        return "\n\n".join(trajectory_parts)

    def get_last_execution(self) -> Optional[str]:
        """
        获取最近一次的执行结果 (例如，最新生成的代码)。
        如果不存在，则返回 None。
        """
        for record in reversed(self.records):
            if record['type'] == 'execution':
                return record['content']
        return None

这个 `Memory` 类的设计比较简洁，主体是这样的：

- 使用一个列表 `records` 来按顺序存储每一次的行动和反思。
- `add_record` 方法负责向记忆中添加新的条目。
- `get_trajectory` 方法是核心，它将记忆轨迹“序列化”成一段文本，可以直接插入到后续的提示词中，为模型的反思和优化提供完整的上下文。
- `get_last_execution` 方便我们获取最新的“初稿”以供反思。



### 4.4.3 Reflection 智能体的编码实现

有了 `Memory` 模块作为基础，我们现在可以着手构建 `ReflectionAgent` 的核心逻辑。整个智能体的工作流程将围绕我们之前讨论的“执行-反思-优化”循环展开，并通过精心设计的提示词来引导大语言模型扮演不同的角色。

（1）提示词设计

与之前的范式不同，Reflection 机制需要多个不同角色的提示词来协同工作。

1. <strong>初始执行提示词 (Execution Prompt)</strong> ：这是智能体首次尝试解决问题的提示词，内容相对直接，只要求模型完成指定任务。


In [17]:
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。请根据以下要求，编写一个Python函数。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。

要求: {task}

请直接输出代码，不要包含任何额外的解释。
"""

2. <strong>反思提示词 (Reflection Prompt)</strong> ：这个提示词是 Reflection 机制的灵魂。它指示模型扮演“代码评审员”的角色，对上一轮生成的代码进行批判性分析，并提供具体的、可操作的反馈。


In [21]:
REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下Python代码，并专注于找出其在<strong>算法效率</strong>上的主要瓶颈。

# 原始任务:
{task}

# 待审查的代码:
```python
{code}
```

请分析该代码的时间复杂度，并思考是否存在一种<strong>算法上更优</strong>的解决方案来显著提升性能。
如果存在，请清晰地指出当前算法的不足，并提出具体的、可行的改进算法建议（例如，使用筛法替代试除法）。
如果代码在算法层面已经达到最优，才能回答“无需改进”。

请直接输出你的反馈，不要包含任何额外的解释。
"""

3. <strong>优化提示词 (Refinement Prompt)</strong> ：当收到反馈后，这个提示词将引导模型根据反馈内容，对原有代码进行修正和优化。

In [18]:
REFINE_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。你正在根据一位代码评审专家的反馈来优化你的代码。

# 原始任务:
{task}

# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈：
{feedback}

请根据评审员的反馈，生成一个优化后的新版本代码。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。
请直接输出优化后的代码，不要包含任何额外的解释。
"""

（2）智能体封装与实现

现在，我们将这套提示词逻辑和 `Memory` 模块整合到 `ReflectionAgent` 类中。

In [22]:
# 假设 llm_client.py 和 memory.py 已定义
# from llm_client import HelloAgentsLLM
# from memory import Memory

class ReflectionAgent:
    def __init__(self, llm_client, max_iterations=3):
        self.llm_client = llm_client
        self.memory = Memory()
        self.max_iterations = max_iterations

    def run(self, task: str):
        print(f"\n--- 开始处理任务 ---\n任务: {task}")

        # --- 1. 初始执行 ---
        print("\n--- 正在进行初始尝试 ---")
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code = self._get_llm_response(initial_prompt)
        self.memory.add_record("execution", initial_code)

        # --- 2. 迭代循环:反思与优化 ---
        for i in range(self.max_iterations):
            print(f"\n--- 第 {i+1}/{self.max_iterations} 轮迭代 ---")

            # a. 反思
            print("\n-> 正在进行反思...")
            last_code = self.memory.get_last_execution()
            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(task=task, code=last_code)
            feedback = self._get_llm_response(reflect_prompt)
            self.memory.add_record("reflection", feedback)

            # b. 检查是否需要停止
            if "无需改进" in feedback:
                print("\n✅ 反思认为代码已无需改进，任务完成。")
                break

            # c. 优化
            print("\n-> 正在进行优化...")
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback
            )
            refined_code = self._get_llm_response(refine_prompt)
            self.memory.add_record("execution", refined_code)
        
        final_code = self.memory.get_last_execution()
        print(f"\n--- 任务完成 ---\n最终生成的代码:\n```python\n{final_code}\n```")
        return final_code

    def _get_llm_response(self, prompt: str) -> str:
        """一个辅助方法，用于调用LLM并获取完整的流式响应。"""
        messages = [{"role": "user", "content": prompt}]
        response_text = self.llm_client.think(messages=messages) or ""
        return response_text

In [23]:
llm_client = HelloAgentsLLM()
reflection_agent = ReflectionAgent(llm_client, 7)
reflection_agent.run("编写一个Python函数，找到1到n之间的所有素数(prime numbers)。")


--- 开始处理任务 ---
任务: 编写一个Python函数，找到1到n之间的所有素数(prime numbers)。

--- 正在进行初始尝试 ---
🧠 正在调用 Qwen/Qwen3.5-35B-A3B 模型...
✅ 大语言模型响应成功:
```python
def find_primes(n: int) -> list[int]:
    """Find all prime numbers between 1 and n inclusive.

    This function uses the Sieve of Eratosthenes algorithm to efficiently
    identify prime numbers up to the specified limit n.

    Args:
        n (int): The upper limit for finding prime numbers.

    Returns:
        list[int]: A list of prime numbers in the range [1, n].
                   Returns an empty list if n < 2.
    """
    if n < 2:
        return []

    # Initialize a boolean list where index represents the number
    # and value represents whether the number is prime.
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False

    # Iterate from 2 to sqrt(n)
    for i in range(2, int(n**0.5) + 1):
        if is_prime[i]:
            # Mark multiples of i starting from i*i
            for j in range(i * i, n + 1, i):
          

'```python\ndef find_primes(n: int) -> list[int]:\n    """Find all prime numbers between 1 and n inclusive using Linear Sieve (Euler Sieve).\n\n    This implementation achieves O(n) time complexity by ensuring each composite\n    number is marked exactly once by its smallest prime factor. It eliminates\n    the redundant marking operations inherent in the Sieve of Eratosthenes.\n\n    Args:\n        n (int): The upper limit for finding prime numbers.\n\n    Returns:\n        list[int]: A list of prime numbers in the range [1, n].\n                   Returns an empty list if n < 2.\n    """\n    if n < 2:\n        return []\n\n    # is_composite[i] indicates whether i is composite (1) or prime (0)\n    # Initialized to 0s (all assumed prime initially)\n    is_composite = bytearray(n + 1)\n    primes: list[int] = []\n    \n    # Cache append method for slight performance gain in tight loops\n    primes_append = primes.append\n\n    for i in range(2, n + 1):\n        if not is_composite[i